# 18 · EHR — trần dự báo và **residual audit**

Nhánh **1b** của kế hoạch (`sleepy-puzzling-harbor.md`). **CPU thuần, không GPU.**

## Câu hỏi

> **EHR có mang thông tin mà model ảnh chưa có không?**

nb11–15 đã thử ba kiến trúc fusion (FiLM/CondAdapter ở block muộn, disease gate, residual gate)
và đều thu được ít. Nhưng chưa notebook nào hỏi câu trên. Chúng đi thẳng từ *"có EHR"* sang
*"dựng kiến trúc"*.

Hệ quả: khi một kiến trúc không ăn thua, ta **không phân biệt được** hai khả năng —

| | |
|---|---|
| (a) | EHR **không có** tín hiệu bổ sung ⇒ mọi kiến trúc đều vô ích |
| (b) | EHR **có** tín hiệu nhưng kiến trúc đó không lấy được ⇒ nên thử kiến trúc khác |

Notebook này tách hai khả năng đó bằng bộ kết hợp **đơn giản nhất có thể**: một hồi quy
logistic hai biến. Nếu **nó** không lấy được gì thì (a) là lời giải thích khả dĩ hơn hẳn.

## Ba bước

| | làm gì |
|---|---|
| `E1` | **Trần EHR-only** — gradient boosting mỗi nhãn, fit trên TRAIN, đánh giá trên val. So với prevalence và với AUROC `0.5` |
| `E2` | **Residual audit** — stacker `[logit(p_img), logit(p_ehr)]`, cross-fit 5-fold theo `subject_id` **trong val** |
| `E3` | Phân rã theo nhãn và theo `setting` (inpatient / ED / outpatient) |

## ⚠ Hai bẫy đã tính trước

### 1. Không fit stacker trên TRAIN

Backbone `04b` và head `H5` đều đã train trên train, nên `p_img` trên train **lạc quan**. Stacker
fit ở đó sẽ tin `p_img` quá mức và hạ thấp vai trò EHR một cách giả tạo. Nên stacker
**cross-fit trong val** — val chưa bao giờ được train.

Model EHR thì fit trên **train** là đúng: nó không dính gì tới model ảnh.

### 2. Mốc so sánh phải là **stacker một biến**, không phải AP ảnh thô

Cross-fit làm mỗi fold có hệ số riêng. Hồi quy logistic một biến là **đơn điệu** nên trong
từng fold nó giữ nguyên thứ hạng — nhưng gộp OOF của 5 fold có **thang khác nhau** lại thì thứ
hạng toàn cục **có thể đổi**. Đó là chi phí của chính bộ khung, không liên quan gì tới EHR.

Nên phép so chính là

```
AP( stacker[logit p_img, logit p_ehr] )  −  AP( stacker[logit p_img] )
```

Hai vế **cùng** bộ khung cross-fit ⇒ chi phí khung bị triệt tiêu, phần còn lại đúng là đóng góp
của EHR. `C4` đo luôn chi phí khung đó và in ra.

## C0 · Config + **cổng quyết định khoá trước**

Cổng viết ra **trước khi chạy**, để một con số nhỏ không bị diễn giải thành tín hiệu — và để
một con số lớn không bị bỏ qua.

### Mốc ảnh: một, và là `R0_ep2` *(chốt 2026-07-31)*

`p_img` dùng **một** mốc duy nhất — `R0_ep2`, đúng checkpoint mà quy tắc §9 của nb16 chọn.

Có cân nhắc dùng ensemble làm mốc bảo thủ, nhưng bỏ: chênh lệch giữa hai mốc chỉ `0.0011`
macro (`0.25%` tương đối), nên độ thổi phồng lên đóng góp của EHR nhỏ hơn hẳn độ rộng CI của
chính phép đo. Thêm một mốc chỉ làm bảng dài ra mà không đổi được kết luận nào.

In [1]:
import os, json, math, time, pickle, re
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
NB16_PRED = os.path.join(DATA_DIR, 'label_reliability', 'preds')
SC_DIR    = os.path.join(DATA_DIR, 'ehr_audit')
os.makedirs(f'{SC_DIR}/metrics', exist_ok=True)

SEED, K_FOLD, N_BOOT = 42, 5, 2000
RARE3 = ['Pleural_Other', 'Fracture', 'Lung_Lesion']
IMG_KEY = 'R0_ep2'                       # model anh tham chieu (= REF_MODEL cua nb17)
TOPK_BAG = 30                            # multi-hot top-K cho moi tui micro

# ---------------- COÅNG KHOA TRUOC ----------------
GATE = dict(
    contrast='AP( stacker[p_img, p_ehr] ) - AP( stacker[p_img] ), CUNG bo khung cross-fit',
    on=['macro', 'rare3'],
    dong='<= +0.001  -> EHR KHONG con dang dau tu; dong nhanh fusion, ghi vao bao cao',
    tiep='>  +0.002  -> co tin hieu du; mo "Fusion EHR moi" (uu tien 4)',
    giua='o giua     -> khong ket luan duoc; KHONG dau tu them',
    ci='moi ket luan phai kem CI subject-cluster; diem don khong du')
PREREG = dict(notebook='18_ehr_residual_audit.ipynb',
              plan='sleepy-puzzling-harbor.md — nhanh 1b',
              ehr_model='fit tren TRAIN (khong dinh model anh)',
              stacker=f'cross-fit {K_FOLD}-fold theo subject_id TRONG val',
              why_not_train='p_img tren train lac quan (04b/H5 da train o do) -> stacker se '
                            'ha thap EHR mot cach gia tao',
              reference='stacker MOT bien [logit p_img] — KHONG phai AP anh tho',
              gate=GATE, img_key=IMG_KEY, n_boot=N_BOOT, no_test_access=True)
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json', 'w'), indent=1)
for k, v in GATE.items(): print(f'  {k:<9} {v}')
print()
print('CPU thuan. Khong dung GPU, khong dung anh.')

Mounted at /content/drive
  contrast  AP( stacker[p_img, p_ehr] ) - AP( stacker[p_img] ), CUNG bo khung cross-fit
  on        ['macro', 'rare3']
  dong      <= +0.001  -> EHR KHONG con dang dau tu; dong nhanh fusion, ghi vao bao cao
  tiep      >  +0.002  -> co tin hieu du; mo "Fusion EHR moi" (uu tien 4)
  giua      o giua     -> khong ket luan duoc; KHONG dau tu them
  ci        moi ket luan phai kem CI subject-cluster; diem don khong du

CPU thuan. Khong dung GPU, khong dung anh.


## C1 · Nhãn + prediction ảnh + bộ đo

`p_img` lấy từ prediction study-level nb16 đã lưu — không cần chạy lại model.

`ap_grouped` copy verbatim từ nb16 `C15` (đã kiểm khớp `sklearn` bit-for-bit kể cả khi có tie).

In [2]:
SID_VAL  = np.load(f'{NB16_PRED}/sid_val.npy')
Y_VAL    = np.load(f'{NB16_PRED}/y_val.npy')
SUBJ_VAL = np.load(f'{NB16_PRED}/subj_val.npy')
P_IMG    = np.load(f'{NB16_PRED}/{IMG_KEY.replace("_ep", f"_s{SEED}_ep")}.npy')
meta = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS)
cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], 'float64'); N_TRAIN = int(cf['num_train'])
TIER = np.where(N_POS_TRAIN/N_TRAIN < 0.02, 'tail',
                np.where(N_POS_TRAIN/N_TRAIN < 0.10, 'medium', 'head'))
RARE3_IDX  = np.array([LABELS.index(l) for l in RARE3])
NONTGT_IDX = np.setdiff1d(np.arange(C), RARE3_IDX)
GRP = dict(macro=np.arange(C), rare3=RARE3_IDX, nontgt=NONTGT_IDX)
n_st = len(SID_VAL)
assert P_IMG.shape == (n_st, C)
print(f'VAL {n_st:,} study | {len(np.unique(SUBJ_VAL)):,} subject | p_img <- {IMG_KEY}')

def _prep(p):
    o = np.argsort(-p, kind='stable'); ps = p[o]
    return o, np.r_[0, np.cumsum(ps[1:] != ps[:-1])].astype('int64'), 0
def _prep2(p):
    o, gid, _ = _prep(p); return o, gid, int(gid[-1])+1
def ap_grouped(yo, co, gid, ng):
    a = np.bincount(gid, weights=yo*co, minlength=ng)
    n = np.bincount(gid, weights=co, minlength=ng)
    npos = a.sum()
    if npos <= 0: return np.nan
    TP = np.cumsum(a); N = np.cumsum(n); m = a > 0
    return float((a[m]/npos*(TP[m]/N[m])).sum())
def ap_vec(PS, y=None):
    y = Y_VAL if y is None else y
    out = np.full(C, np.nan)
    for j in range(C):
        if y[:, j].sum() == 0: continue
        o, gid, ng = _prep2(PS[:, j])
        out[j] = ap_grouped(y[o, j].astype('float64'), np.ones(len(o)), gid, ng)
    return out
_ref = np.array([average_precision_score(Y_VAL[:, j], P_IMG[:, j]) for j in range(C)])
assert np.max(np.abs(ap_vec(P_IMG)-_ref)) < 1e-9, 'ap_grouped lech sklearn'
print('ap_grouped == sklearn  OK')

_SU = np.unique(SUBJ_VAL); _six = np.searchsorted(_SU, SUBJ_VAL); _nsu = len(_SU)
_rg = np.random.default_rng(SEED)
BOOT_C = [np.bincount(_rg.integers(0, _nsu, _nsu), minlength=_nsu)[_six] for _ in range(N_BOOT)]
def boot_grp(PS):
    pr = [_prep2(PS[:, j]) for j in range(C)]
    yo = [Y_VAL[pr[j][0], j].astype('float64') for j in range(C)]
    out = {g: np.empty(N_BOOT) for g in GRP}
    for b, c in enumerate(BOOT_C):
        a = np.array([ap_grouped(yo[j], c[pr[j][0]].astype('float64'), pr[j][1], pr[j][2])
                      for j in range(C)])
        for g, jj in GRP.items(): out[g][b] = np.nanmean(a[jj])
    return out
def delta_ci(bA, bB, pA, pB, g):
    d = bA[g]-bB[g]; lo, hi = np.percentile(d, [2.5, 97.5])
    return (float(np.nanmean(pA[GRP[g]])-np.nanmean(pB[GRP[g]])), float(lo), float(hi),
            bool(lo > 0 or hi < 0))

def subject_folds(subj, K=K_FOLD, seed=SEED):
    su = np.unique(subj); r = np.random.default_rng(seed)
    f = np.empty(len(su), 'int64'); f[r.permutation(len(su))] = np.arange(len(su)) % K
    return pd.Series(subj).map(dict(zip(su, f))).to_numpy()
FOLD = subject_folds(SUBJ_VAL)
for k in range(K_FOLD):
    assert set(SUBJ_VAL[FOLD == k]).isdisjoint(set(SUBJ_VAL[FOLD != k]))
print(f'cross-fit {K_FOLD} fold theo subject: OK, roi nhau')
LOGIT = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6)/(1-np.clip(p, 1e-6, 1-1e-6)))

VAL 31,670 study | 8,956 subject | p_img <- R0_ep2
ap_grouped == sklearn  OK
cross-fit 5 fold theo subject: OK, roi nhau


## C2 · Dựng đặc trưng EHR — theo đúng nb13 cell 6

Cùng một đường dựng đã dùng qua 5 notebook (`06b`, `11`, `13`, `14`, `15`), nên nếu EHR có tín
hiệu thì nó phải nằm trong đúng bộ đặc trưng này.

| nhóm | nguồn |
|---|---|
| labs | `{c}_value` / `{c}_mask` với `c ∈ vocab['labs']` |
| vitals | `{c}_value` / `{c}_mask` |
| comorbidity | `vocab['comorbidity']` + `como_mask` |
| counts | `t2_n_pending`, `t3_n_specimens`, `t4_n_prior_positive_ever`, `t4_years_since_first_culture`, `t4_n_prior_encounters` |
| binary | `t2_pending`, `t2_time_imprecise`, `t1_same_encounter` |
| categorical | `setting`, `t3_recency_bin`, `t1_status` |
| recency | `t1_days_since_last_specimen`, `t1_days_since_last_positive`, `t3_days_since_last_positive` |
| micro (ragged) | `t1_org`, `t1_spec`, `t2_spec`, `t3_org` → **multi-hot top-30** + đếm |

Bốn túi micro là danh sách id độ dài thay đổi. Cây quyết định không ăn được danh sách, nên mã
hoá thành multi-hot của **30 id thường gặp nhất trong TRAIN** cộng một biến đếm. Top-K chọn
trên **train**, không phải trên val.

Z-score cũng dùng thống kê **train** — y hệt nb13.

In [3]:
ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
MISSING = vocab['reserved']['MISSING']
tr_mask = (ehr.split == 'train').to_numpy()
va_mask = (ehr.split == 'val').to_numpy()
print(f'cohort {len(ehr):,} | train {tr_mask.sum():,} | val {va_mask.sum():,}')

FEAT, NAMES = [], []
def add(block, names):
    b = np.asarray(block, 'float32')
    if b.ndim == 1: b = b[:, None]
    assert b.shape[0] == len(ehr) and b.shape[1] == len(names), (b.shape, len(names))
    FEAT.append(b); NAMES.extend(names)

LABS = vocab['labs']
add(np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in LABS], 1), [f'lab_{c}' for c in LABS])
add(np.stack([ehr[f'{c}_mask'].to_numpy('float32') for c in LABS], 1), [f'labm_{c}' for c in LABS])
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr.columns]
if VITALS:
    add(np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in VITALS], 1), [f'vit_{c}' for c in VITALS])
    add(np.stack([ehr[f'{c}_mask'].to_numpy('float32') for c in VITALS], 1), [f'vitm_{c}' for c in VITALS])
COMO = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr.columns]
if COMO: add(np.stack([ehr[c].to_numpy('float32') for c in COMO], 1), [f'como_{c}' for c in COMO])

def zscore_train(x, use_log=True):
    z = np.log1p(np.clip(x.astype('float32'), 0, None)) if use_log else x.astype('float32')
    mu, sd = z[tr_mask].mean(), z[tr_mask].std() or 1.0
    return ((z-mu)/sd).astype('float32')
COUNTS = ['t2_n_pending', 't3_n_specimens', 't4_n_prior_positive_ever',
          't4_years_since_first_culture', 't4_n_prior_encounters']
add(np.stack([zscore_train(ehr[c].to_numpy()) for c in COUNTS], 1), [f'cnt_{c}' for c in COUNTS])
BIN = ['t2_pending', 't2_time_imprecise', 't1_same_encounter']
add(np.stack([ehr[b].to_numpy('float32') for b in BIN], 1), [f'bin_{b}' for b in BIN])
def codes(series, cats):
    m = {c.lower(): i for i, c in enumerate(cats)}
    return series.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('float32')
add(np.stack([codes(ehr['setting'], ['inpatient', 'ED', 'outpatient']),
              codes(ehr['t3_recency_bin'], ['none', '7-30d', '30-90d']),
              ehr['t1_status'].to_numpy('float32')], 1), ['cat_setting', 'cat_recency', 'cat_status'])
REC1 = ['t1_days_since_last_specimen', 't1_days_since_last_positive']
REC3 = ['t3_days_since_last_positive']
for grp in (REC1, REC3):
    add(np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in grp], 1), [f'rec_{c}' for c in grp])
    add(np.stack([ehr[f'{c}_mask'].to_numpy('float32') for c in grp], 1), [f'recm_{c}' for c in grp])

# ---------------- micro ragged -> multi-hot top-K (chon tren TRAIN) ----------------
RAGGED = {'t1_org': 't1_org_ids', 't1_spec': 't1_spec_ids',
          't2_spec': 't2_spec_ids', 't3_org': 't3_org_ids'}
for nm, col in RAGGED.items():
    lists = [np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
             for v in ehr[col].to_numpy()]
    lists = [a[a != MISSING] for a in lists]
    from collections import Counter
    cnt = Counter()
    for i in np.where(tr_mask)[0]: cnt.update(set(lists[i].tolist()))
    top = [i for i, _ in cnt.most_common(TOPK_BAG)]
    M = np.zeros((len(ehr), len(top)+1), 'float32')
    pos = {v: k for k, v in enumerate(top)}
    for i, a in enumerate(lists):
        for v in set(a.tolist()):
            if v in pos: M[i, pos[v]] = 1.0
        M[i, -1] = len(a)
    add(M, [f'{nm}_id{v}' for v in top] + [f'{nm}_n'])
    print(f'  {nm}: {len(top)} id top-K tren train (tren tong {len(cnt)}) + 1 dem')

X = np.concatenate(FEAT, 1)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
Y_ALL = ehr[LABELS].to_numpy('float32')
assert len(NAMES) == X.shape[1]
print()
print(f'ma tran dac trung EHR: {X.shape}  ({X.shape[1]} cot)')
# val cua EHR phai trung thu tu study voi p_img
_sid_va = ehr.loc[va_mask, 'study_id'].to_numpy()
assert np.array_equal(np.sort(_sid_va), SID_VAL), 'thu tu study val cua EHR != nb16'
_ord = np.argsort(_sid_va, kind='stable')
X_VA = X[va_mask][_ord]; Y_VA = Y_ALL[va_mask][_ord]
assert np.array_equal(Y_VA, Y_VAL), 'nhan val cua EHR != nhan cua nb16'
X_TR, Y_TR = X[tr_mask], Y_ALL[tr_mask]
print(f'  train {X_TR.shape} | val {X_VA.shape}  (thu tu study da khop nb16)')

cohort 211,130 | train 147,791 | val 31,670
  t1_org: 30 id top-K tren train (tren tong 262) + 1 dem
  t1_spec: 30 id top-K tren train (tren tong 74) + 1 dem
  t2_spec: 30 id top-K tren train (tren tong 73) + 1 dem
  t3_org: 30 id top-K tren train (tren tong 285) + 1 dem

ma tran dac trung EHR: (211130, 170)  (170 cot)
  train (147791, 170) | val (31670, 170)  (thu tu study da khop nb16)


## C3 · **E1 — trần EHR-only**

Gradient boosting mỗi nhãn, **fit trên TRAIN**, đánh giá trên val.

Hai thước, đọc cùng nhau:

| | ý nghĩa |
|---|---|
| `AP` vs **prevalence** | prevalence là AP của một bộ đoán ngẫu nhiên; `AP ≈ prevalence` ⇒ không có gì |
| `AUROC` vs **0.5** | bất biến với prevalence ⇒ so được giữa nhãn hiếm và nhãn thường |

Với nhãn hiếm (`Pleural_Other` prevalence `~1.2%`), `AP = 0.03` nghe như không có gì nhưng thực
ra là **gấp 2.5× ngẫu nhiên**. Nên `AUROC` là thước dễ đọc hơn ở đây, còn `AP` là thước mà cả
chương trình đang tối ưu.

⚠ `E1` **không** quyết định gì một mình. EHR có thể dự báo tốt mà vẫn **thừa** so với ảnh — đó
đúng là việc của `E2`.

In [4]:
try:
    from xgboost import XGBClassifier
    MK = lambda: XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.08,
                               subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                               n_jobs=-1, random_state=SEED, eval_metric='logloss',
                               tree_method='hist')
    ENGINE = 'xgboost'
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    MK = lambda: HistGradientBoostingClassifier(max_iter=300, max_depth=6, learning_rate=0.08,
                                                random_state=SEED)
    ENGINE = 'sklearn HistGradientBoosting'
print(f'engine: {ENGINE}')

EHR_NPY = f'{SC_DIR}/p_ehr_val_{IMG_KEY}.npy'
if os.path.exists(EHR_NPY):
    P_EHR = np.load(EHR_NPY); print('p_ehr <- cache')
else:
    P_EHR = np.zeros((n_st, C), 'float32'); t0 = time.time()
    for j in tqdm(range(C), desc='E1 fit EHR-only tren TRAIN'):
        m = MK().fit(X_TR, Y_TR[:, j])
        P_EHR[:, j] = m.predict_proba(X_VA)[:, 1]
    np.save(EHR_NPY, P_EHR); print(f'xong sau {(time.time()-t0)/60:.1f} phut')

PREV = Y_VAL.mean(0)
ap_e, ap_i = ap_vec(P_EHR), ap_vec(P_IMG)
au_e = np.array([roc_auc_score(Y_VAL[:, j], P_EHR[:, j]) for j in range(C)])
au_i = np.array([roc_auc_score(Y_VAL[:, j], P_IMG[:, j]) for j in range(C)])
E1 = pd.DataFrame(dict(label=LABELS, tier=TIER, prevalence=PREV,
                       AP_ehr=ap_e, AP_img=ap_i, AP_ehr_over_prev=ap_e/np.maximum(PREV, 1e-9),
                       AUROC_ehr=au_e, AUROC_img=au_i))
print()
print('=== E1 · TRAN EHR-ONLY (fit TRAIN, danh gia VAL) ===')
print(E1.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
E1.to_csv(f'{SC_DIR}/metrics/ehr_only.csv', index=False)
print()
for g, jj in GRP.items():
    print(f'  {g:<7} AP  ehr {np.nanmean(ap_e[jj]):.4f} vs img {np.nanmean(ap_i[jj]):.4f} '
          f'vs prevalence {PREV[jj].mean():.4f}   |  AUROC ehr {au_e[jj].mean():.4f} vs img {au_i[jj].mean():.4f}')
_w = E1[E1.AUROC_ehr < 0.55]
print()
print(f'  nhan co AUROC(EHR) < 0.55 (gan nhu khong tin hieu): '
      f'{list(_w.label) if len(_w) else "khong co"}')
assert (au_e[TIER == "head"] > 0.5).any(), \
    'KHONG nhan head nao co AUROC(EHR) > 0.5 -> pipeline dac trung hong, khong phai EHR vo dung'
print('  gate: it nhat mot nhan head co AUROC(EHR) > 0.5  OK (pipeline dac trung chay dung)')

engine: xgboost


E1 fit EHR-only tren TRAIN:   0%|          | 0/13 [00:00<?, ?it/s]

xong sau 3.8 phut

=== E1 · TRAN EHR-ONLY (fit TRAIN, danh gia VAL) ===
                     label   tier  prevalence  AP_ehr  AP_img  AP_ehr_over_prev  AUROC_ehr  AUROC_img
               Atelectasis   head      0.2502  0.3851  0.5779            1.5391     0.6860     0.8248
              Cardiomegaly   head      0.2254  0.4012  0.5420            1.7802     0.7323     0.8175
             Consolidation medium      0.0672  0.1343  0.2094            1.9987     0.7028     0.7937
                     Edema   head      0.1797  0.3728  0.6410            2.0746     0.7523     0.8893
Enlarged_Cardiomediastinum medium      0.0734  0.1367  0.1739            1.8629     0.7118     0.7370
                  Fracture medium      0.0218  0.0469  0.1085            2.1509     0.6749     0.7770
               Lung_Lesion medium      0.0322  0.0582  0.2125            1.8082     0.6509     0.8274
              Lung_Opacity   head      0.2448  0.3715  0.5055            1.5174     0.6681     0.7840
          

## C4 · **E2 — residual audit** *(phép thử quyết định)*

Ba stacker, **cùng một bộ khung cross-fit**, khác nhau đúng ở tập biến đầu vào:

| | biến | vai trò |
|---|---|---|
| `S_img` | `[logit p_img]` | **mốc** — hấp thụ chi phí của chính bộ khung |
| `S_ehr` | `[logit p_ehr]` | tham chiếu |
| `S_both` | `[logit p_img, logit p_ehr]` | cái đang xét |

Phép so quyết định là **`S_both − S_img`**. Cả hai vế cùng chịu chi phí cross-fold nên nó bị
triệt tiêu, phần còn lại đúng là đóng góp của EHR.

Cell in luôn **chi phí khung** (`S_img − p_img thô`) để anh thấy nó lớn cỡ nào. Nếu chi phí đó
so được với hiệu ứng đang tìm thì phải nói rõ trong báo cáo.

In [5]:
def cross_fit_stacker(cols):
    """cols: list cac ma tran [n_st, C] o khong gian logit. Tra ve OOF [n_st, C]."""
    OOF = np.full((n_st, C), np.nan); seen = np.zeros(n_st, 'int64')
    for k in range(K_FOLD):
        tr = np.where(FOLD != k)[0]; te = np.where(FOLD == k)[0]
        for j in range(C):
            Xtr = np.column_stack([c[tr, j] for c in cols])
            Xte = np.column_stack([c[te, j] for c in cols])
            y = Y_VAL[tr, j]
            if y.sum() < 5 or y.sum() == len(tr):
                OOF[te, j] = cols[0][te, j]                      # khong fit duoc -> giu bien dau
            else:
                OOF[te, j] = LogisticRegression(max_iter=2000).fit(Xtr, y).predict_proba(Xte)[:, 1]
        seen[te] += 1
    assert (seen == 1).all() and np.isfinite(OOF).all()
    return OOF

LI, LE = LOGIT(P_IMG), LOGIT(P_EHR)
S = dict(S_img=cross_fit_stacker([LI]),
         S_ehr=cross_fit_stacker([LE]),
         S_both=cross_fit_stacker([LI, LE]))
AP = {k: ap_vec(v) for k, v in S.items()}; AP['p_img_tho'] = ap_i; AP['p_ehr_tho'] = ap_e
B  = {k: boot_grp(v) for k, v in S.items()}; B['p_img_tho'] = boot_grp(P_IMG)

print('=== AP theo nhom ===')
for k in ('p_img_tho', 'S_img', 'p_ehr_tho', 'S_ehr', 'S_both'):
    print(f'  {k:<11} ' + ' | '.join(f'{g} {np.nanmean(AP[k][jj]):.4f}' for g, jj in GRP.items()))

print()
print('=== CHI PHI CUA BO KHUNG: S_img - p_img tho ===')
for g in GRP:
    d, lo, hi, ex = delta_ci(B['S_img'], B['p_img_tho'], AP['S_img'], AP['p_img_tho'], g)
    print(f'  {g:<7} {d:+.5f}  CI95 [{lo:+.5f}, {hi:+.5f}]  {"LOAI 0" if ex else "chua 0"}')
print('  (khac 0 la do 5 fold co thang khac nhau khi gop OOF — khong lien quan toi EHR)')

print()
print('=== E2 · PHEP THU QUYET DINH:  S_both - S_img ===')
rows = []
for g in GRP:
    d, lo, hi, ex = delta_ci(B['S_both'], B['S_img'], AP['S_both'], AP['S_img'], g)
    rows.append(dict(group=g, delta=d, ci_lo=lo, ci_hi=hi, excl0=ex))
    print(f'  {g:<7} {d:+.5f}  CI95 [{lo:+.5f}, {hi:+.5f}]  {"LOAI 0" if ex else "chua 0"}')
E2 = pd.DataFrame(rows); E2.to_csv(f'{SC_DIR}/metrics/residual_audit.csv', index=False)

print()
print('=== COÅNG (khoa o C0) ===')
_verdict = {}
for g in ('macro', 'rare3'):
    r = E2[E2.group == g].iloc[0]
    v = ('DONG'  if r.delta <= 0.001 else
         'TIEP'  if r.delta > 0.002 else 'GIUA')
    _verdict[g] = v
    print(f'  {g:<7} delta {r.delta:+.5f} -> ' +
          {'DONG': 'EHR KHONG con dang dau tu',
           'TIEP': 'co tin hieu du -> mo Fusion EHR moi',
           'GIUA': 'KHONG ket luan duoc -> khong dau tu them'}[v] +
          (f'   (CI loai 0)' if r.excl0 else '   (CI chua 0)'))
print()
if all(v == 'DONG' for v in _verdict.values()):
    print('  => KET LUAN: bo ket hop DON GIAN NHAT khong lay duoc gi tu EHR.')
    print('     Mot kien truc fusion phuc tap hon kho lay duoc hon. Dong nhanh nay.')
    print('     Day la cau tra loi ma nb11-15 KHONG the cho, vi chung chi thu tung kien truc.')
elif any(v == 'TIEP' for v in _verdict.values()):
    print('  => KET LUAN: CO tin hieu du du mot logistic 2 bien cung lay duoc.')
    print('     Kien truc fusion tot hon co the lay duoc nhieu hon -> uu tien 4 dang gia.')
else:
    print('  => KET LUAN: khong ket luan duoc. Ghi vao bao cao la KHONG KET LUAN,')
    print('     khong dien giai theo huong nao. Khong dau tu them.')

=== AP theo nhom ===
  p_img_tho   macro 0.4365 | rare3 0.1467 | nontgt 0.5234
  S_img       macro 0.4360 | rare3 0.1453 | nontgt 0.5231
  p_ehr_tho   macro 0.2732 | rare3 0.0434 | nontgt 0.3422
  S_ehr       macro 0.2724 | rare3 0.0416 | nontgt 0.3417
  S_both      macro 0.4385 | rare3 0.1480 | nontgt 0.5256

=== CHI PHI CUA BO KHUNG: S_img - p_img tho ===
  macro   -0.00051  CI95 [-0.00072, -0.00027]  LOAI 0
  rare3   -0.00139  CI95 [-0.00224, -0.00040]  LOAI 0
  nontgt  -0.00025  CI95 [-0.00034, -0.00016]  LOAI 0
  (khac 0 la do 5 fold co thang khac nhau khi gop OOF — khong lien quan toi EHR)

=== E2 · PHEP THU QUYET DINH:  S_both - S_img ===
  macro   +0.00251  CI95 [+0.00151, +0.00349]  LOAI 0
  rare3   +0.00269  CI95 [+0.00009, +0.00539]  LOAI 0
  nontgt  +0.00245  CI95 [+0.00143, +0.00348]  LOAI 0

=== COÅNG (khoa o C0) ===
  macro   delta +0.00251 -> co tin hieu du -> mo Fusion EHR moi   (CI loai 0)
  rare3   delta +0.00269 -> co tin hieu du -> mo Fusion EHR moi   (CI loai 0)



## C5 · **E3** — phân rã theo nhãn và theo `setting`

Chạy dù `E2` ra kết quả nào: nếu `E2` dương thì `E3` cho biết **ở đâu**; nếu `E2` null thì `E3`
kiểm xem có nhóm con nào bị số gộp che mất không.

`setting` đáng tách vì EHR của bệnh nhân **nội trú** dày hơn hẳn ngoại trú — nếu EHR có tác
dụng thì đó là nơi dễ thấy nhất.

In [6]:
d_lab = ap_vec(S['S_both']) - ap_vec(S['S_img'])
E3 = pd.DataFrame(dict(label=LABELS, tier=TIER, prevalence=PREV,
                       AP_S_img=ap_vec(S['S_img']), AP_S_both=ap_vec(S['S_both']),
                       delta=d_lab, AUROC_ehr=au_e, rare3=[l in RARE3 for l in LABELS])
                  ).sort_values('delta', ascending=False)
print('=== E3a · delta theo TUNG NHAN (S_both - S_img) ===')
print(E3.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
E3.to_csv(f'{SC_DIR}/metrics/per_label.csv', index=False)

_sv = ehr.loc[va_mask, 'setting'].to_numpy()[_ord]
print()
print('=== E3b · theo SETTING ===')
rows = []
for st in pd.unique(_sv):
    m = _sv == st
    if m.sum() < 500: continue
    a_i = np.array([average_precision_score(Y_VAL[m, j], S['S_img'][m, j])
                    if Y_VAL[m, j].sum() else np.nan for j in range(C)])
    a_b = np.array([average_precision_score(Y_VAL[m, j], S['S_both'][m, j])
                    if Y_VAL[m, j].sum() else np.nan for j in range(C)])
    rows.append(dict(setting=st, n_study=int(m.sum()),
                     **{f'{g}_delta': float(np.nanmean(a_b[jj])-np.nanmean(a_i[jj]))
                        for g, jj in GRP.items()}))
E3b = pd.DataFrame(rows)
print(E3b.to_string(index=False, float_format=lambda v: f'{v:+.5f}'))
E3b.to_csv(f'{SC_DIR}/metrics/by_setting.csv', index=False)
print()
print('  LUU Y: bang setting KHONG co CI va la phan ra HAU NGHIEM. Doc nhu mo ta,')
print('  khong dung de lat nguoc ket luan cua E2.')

json.dump(dict(engine=ENGINE, img_key=IMG_KEY,
               ehr_only={g: float(np.nanmean(ap_e[jj])) for g, jj in GRP.items()},
               ehr_only_auroc={g: float(au_e[jj].mean()) for g, jj in GRP.items()},
               framework_cost={g: delta_ci(B['S_img'], B['p_img_tho'], AP['S_img'],
                                           AP['p_img_tho'], g)[0] for g in GRP},
               residual=E2.to_dict('records'), verdict=_verdict,
               per_label=E3.to_dict('records'), by_setting=E3b.to_dict('records')),
          open(f'{SC_DIR}/metrics/summary.json', 'w'), indent=1)
print()
print(f'-> {SC_DIR}/metrics/summary.json')

=== E3a · delta theo TUNG NHAN (S_both - S_img) ===
                     label   tier  prevalence  AP_S_img  AP_S_both   delta  AUROC_ehr  rare3
                  Fracture medium      0.0218    0.1080     0.1197  0.0118     0.6749   True
                 Pneumonia   head      0.1529    0.4406     0.4481  0.0076     0.6530  False
              Cardiomegaly   head      0.2254    0.5418     0.5471  0.0053     0.7323  False
                No_Finding   head      0.3278    0.7908     0.7944  0.0036     0.7841  False
              Pneumothorax medium      0.0512    0.5649     0.5682  0.0032     0.8062  False
             Consolidation medium      0.0672    0.2092     0.2118  0.0026     0.7028  False
                     Edema   head      0.1797    0.6408     0.6421  0.0013     0.7523  False
               Lung_Lesion medium      0.0322    0.2108     0.2119  0.0011     0.6509   True
              Lung_Opacity   head      0.2448    0.5054     0.5060  0.0006     0.6681  False
Enlarged_Cardiomed